# FastAPI — Senior Python Interview Practice\n\nFour interview-style exercises spanning implementation, trade-offs, and production concerns.\n\n**How to use:** attempt each prompt first, then run and critique the reference solution. Discuss trade-offs aloud as you would in a senior-level interview.


## 1. Validated inference endpoint\n\n### Problem statement\nCreate a FastAPI endpoint that validates a request and delegates to a service.


In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

app = FastAPI()
class GenerateRequest(BaseModel):
    prompt: str = Field(min_length=1, max_length=8000)
    max_tokens: int = Field(default=256, ge=1, le=2048)

@app.post('/generate')
async def generate(request: GenerateRequest):
    if 'forbidden' in request.prompt.lower():
        raise HTTPException(400, 'unsupported request')
    return {'text': request.prompt[:request.max_tokens], 'model': 'demo'}


### Complexity\n- **Time:** Dominated by model call\n- **Space:** Request/output dependent\n\n### Interview tip\nKeep request validation at the boundary and business logic in a testable service.\n\n### Follow-up questions\n- Where do authentication, rate limits, and content safety belong?


## 2. Dependency injection\n\n### Problem statement\nUse FastAPI dependencies to provide a request-scoped service.


In [ ]:
from fastapi import Depends

class SearchService:
    async def search(self, query): return [query]

def get_search_service(): return SearchService()

@app.get('/search')
async def search(q: str, service: SearchService = Depends(get_search_service)):
    return {'results': await service.search(q)}


### Complexity\n- **Time:** Service dependent\n- **Space:** O(1) framework overhead\n\n### Interview tip\nDependencies make overrides easy in tests and centralize lifecycle decisions.\n\n### Follow-up questions\n- How would you use an async lifespan to initialize shared clients?


## 3. Streaming response\n\n### Problem statement\nStream generated token strings as plain text.


In [ ]:
from fastapi.responses import StreamingResponse

async def tokens(prompt):
    for token in prompt.split():
        yield token + ' '

@app.get('/stream')
async def stream(prompt: str):
    return StreamingResponse(tokens(prompt), media_type='text/plain')


### Complexity\n- **Time:** O(tokens)\n- **Space:** O(1) beyond generator\n\n### Interview tip\nStreaming lowers time-to-first-token but complicates errors after headers are sent.\n\n### Follow-up questions\n- When would SSE or WebSockets be a better transport?


## 4. Health endpoints\n\n### Problem statement\nExpose liveness and readiness endpoints; readiness checks a dependency.


In [ ]:
@app.get('/health/live')
async def live(): return {'status': 'ok'}

@app.get('/health/ready')
async def ready():
    # Replace with a lightweight dependency probe.
    return {'status': 'ready'}


### Complexity\n- **Time:** O(check cost)\n- **Space:** O(1)\n\n### Interview tip\nLiveness should not depend on optional upstreams; readiness can.\n\n### Follow-up questions\n- Which checks are safe to run per request versus periodically?
